In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import json, math, time, os
from pathlib import Path
from dataclasses import dataclass

In [2]:
%pip install -q transformers datasets accelerate einops sentencepiece


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
OUT = '/content/drive/MyDrive/flip'
os.makedirs(OUT, exist_ok=True)
print('Output folder:', OUT)

In [3]:
DEVICE = torch.device('cuda')

In [4]:
def quantize_weights(w):
    # absmean scaling
    scale = w.abs().mean().clamp(min=1e-8)
    return w.div(scale).round().clamp(-1, 1).mul(scale)

In [5]:
def quantize_activations(x):
    # activation quant - int8
    scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-8) / 127.0
    return (x / scale).round().clamp(-128, 127) * scale

In [ ]:
class BitLinear(nn.Linear):
    def __init__(self, in_features, out_features):
        super().__init__(in_features, out_features, bias=False)

    def forward(self, x):
        return F.linear(
            quantize_activations(x),
            quantize_weights(self.weight)
        )

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim = True).add(self.eps).rsqrt()
        return (x.float() * norm).to(x.dtype) * self.weight

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq=512, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq).float()
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos", emb.cos())
        self.register_buffer("sin", emb.sin())

    def forward(self, seq_len):
        return self.cos[:seq_len], self.sin[:seq_len]

In [ ]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

In [ ]:
def apply_rope(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden, n_heads, n_kv_heads, head_dim, max_seq=512):
        super().__init__()
        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim   = head_dim
        self.kv_repeat  = n_heads // n_kv_heads

        self.q = BitLinear(hidden, n_heads    * head_dim)
        self.k = BitLinear(hidden, n_kv_heads * head_dim)
        self.v = BitLinear(hidden, n_kv_heads * head_dim)
        self.o = BitLinear(n_heads * head_dim, hidden)

        self.norm = RMSNorm(hidden)
        self.rope = RotaryEmbedding(head_dim, max_seq)

    def forward(self, x):
        B, T, _ = x.shape
        xn = self.norm(x)

        q = self.q(xn).view(B, T, self.n_heads,    self.head_dim).transpose(1, 2)
        k = self.k(xn).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v(xn).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        k = k.repeat_interleave(self.kv_repeat, dim=1)
        v = v.repeat_interleave(self.kv_repeat, dim=1)

        cos, sin = self.rope(T)
        q, k = apply_rope(q, k, cos, sin)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o(out)

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden, ffn_hidden):
        super().__init__()
        self.gate = BitLinear(hidden, ffn_hidden)
        self.up   = BitLinear(hidden, ffn_hidden)
        self.down = BitLinear(ffn_hidden, hidden)
        self.norm = RMSNorm(hidden)

    def forward(self, x):
        xn = self.norm(x)
        return self.down(F.silu(self.gate(xn)) * self.up(xn))

In [ ]:
class Block(nn.Module):
    def __init__(self, hidden, n_heads, n_kv_heads, head_dim, ffn_hidden, max_seq=512):
        super().__init__()
        self.attn = Attention(hidden, n_heads, n_kv_heads, head_dim, max_seq)
        self.mlp  = MLP(hidden, ffn_hidden)

    def forward(self, x):
        x = x + self.attn(x)
        x = x + self.mlp(x)
        return x

<!-- attention block -->